# Проект 1. Почему в апреле 2026 упала выручка от подписок

**Автор:** Рубан Ирина, БАЦРФ262 · курс «Python для бизнес-аналитики», ВШБ НИУ ВШЭ

План: подключаюсь к базе → смотрю на данные и ищу вопрос (раздел 0) → формулирую
вопрос и гипотезы → строю витрину и проверяю её → раскладываю изменение выручки →
вывод и рекомендация.

## Подключение

Функция `q()` — как на семинаре 6: выполняет запрос и сразу закрывает соединение.
Внутри `set time zone 'UTC'`: месяцы и сутки считаем по UTC, тогда границы месяцев
у всех одинаковые.

In [ ]:
import math

import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from matplotlib.ticker import FuncFormatter
from sqlalchemy import text

from prime.config import get_engine

engine = get_engine()

ACCENT = "#C1272D"   # один акцентный цвет — то, о чём вывод
GRAY = "#9CA3AF"     # всё остальное — серым

# Единый стиль всех графиков: seaborn поверх matplotlib, крупные подписи.
sns.set_theme(style="whitegrid", context="notebook", font_scale=1.05)
# Формат чисел на осях: 1 234 567 — тысячи через пробел.
THOUSANDS = FuncFormatter(lambda v, _: f"{v:,.0f}".replace(",", " ").replace("-", "−"))


def q(sql: str, **params) -> pd.DataFrame:
    # Выполнить запрос и вернуть DataFrame. Соединение закрывается сразу.
    try:
        with engine.connect() as conn:
            conn.execute(text("set time zone 'UTC'"))
            return pd.read_sql(text(sql), conn, params=params)
    finally:
        engine.dispose()


def rub(x: float) -> str:
    # 12345678.9 -> '12 345 679'
    return f"{x:,.0f}".replace(",", " ")


print("подключились как:", q("select current_user as login").iloc[0, 0])

## 0. Разведка: как я искала вопрос

**С чего начала.** ПРАЙМ зарабатывает на абонентской плате, комиссии с покупок
в партнёрских сервисах и росте транзакций. Главная и самая прямая строка — **выручка
от подписок**. Но выручка — итог: она зависит от того, сколько подписчиков, сколько
приходит и уходит, на каком тарифе и сколько они платят. Поэтому до вопроса
посмотрела все эти метрики по месяцам за всю историю (январь 2025 — август 2026):
если где-то есть резкое изменение, про него и стоит задавать вопрос.

| Метрика | Как считаю |
|---|---|
| активные на начало месяца (`active_start`) | подписка началась до 1-го числа и не закончилась до него |
| активные в месяце (`active`) | началась до конца месяца и не закончилась до его начала (как на Л5/С6) |
| новые (`new_subs`), доля триалов | `started_at` в этом месяце; `is_trial` среди новых |
| отменённые (`ended`) и отток (`churn_pct`) | `ended_at` в этом месяце; отток = отменённые / активные на начало месяца |
| доля Расширенного (`ext_share_pct`) | активные на тарифе 2 / все активные |
| платежи, доля неуспешных (`failed_pct`) | `success` — платёж; `failed` / все попытки списания |
| средний платёж, ARPU | выручка / число платежей; выручка / активные подписки |
| выручка в месяц и в день | сумма `success`; в день — делю на число дней в месяце |

In [ ]:
# Шаг 1. Подписки по месяцам: сколько было, сколько пришло, сколько ушло.
# Для каждого месяца проверяю каждую подписку условиями из filter.
subs_m = q('''
    select to_char(m.month_start, 'YYYY-MM') as month,
           -- активные на начало месяца: начались раньше и не закончились до 1-го числа
           count(*) filter (where s.started_at < m.month_start
                              and (s.ended_at is null or s.ended_at >= m.month_start)) as active_start,
           -- активные в месяце (как на Л5/С6): начались до конца месяца и не закончились до начала
           count(*) filter (where s.started_at < m.month_start + interval '1 month'
                              and (s.ended_at is null or s.ended_at >= m.month_start)) as active,
           -- новые: оформлены в этом месяце
           count(*) filter (where s.started_at >= m.month_start
                              and s.started_at < m.month_start + interval '1 month') as new_subs,
           count(*) filter (where s.started_at >= m.month_start
                              and s.started_at < m.month_start + interval '1 month'
                              and s.is_trial) as new_trial,
           -- отменённые: закончились в этом месяце
           count(*) filter (where s.ended_at >= m.month_start
                              and s.ended_at < m.month_start + interval '1 month') as ended,
           -- активные на Расширенном тарифе
           count(*) filter (where s.tariff_id = 2
                              and s.started_at < m.month_start + interval '1 month'
                              and (s.ended_at is null or s.ended_at >= m.month_start)) as active_ext
    from generate_series(date '2025-01-01', date '2026-08-01', interval '1 month') as m(month_start)
    cross join prime.subscriptions s
    group by 1
    order by 1
''').set_index("month")

# Шаг 2. Платежи по месяцам: выручка — только success; неуспешные считаю отдельно.
pay_m = q('''
    select to_char(paid_at, 'YYYY-MM') as month,
           count(*) filter (where status = 'success') as n_payments,
           sum(amount) filter (where status = 'success') as revenue,
           count(*) filter (where status = 'failed') as n_failed,
           count(*) as n_attempts
    from prime.payments
    group by 1
    order by 1
''').set_index("month")

panel = subs_m.join(pay_m)
panel["days"] = pd.PeriodIndex(panel.index, freq="M").days_in_month

# Шаг 3. Метрики из этих чисел.
panel["churn_pct"] = panel["ended"] / panel["active_start"] * 100          # отток за месяц, %
panel["trial_share_pct"] = panel["new_trial"] / panel["new_subs"] * 100    # доля триалов среди новых
panel["ext_share_pct"] = panel["active_ext"] / panel["active"] * 100       # доля Расширенного
panel["failed_pct"] = panel["n_failed"] / panel["n_attempts"] * 100        # доля неуспешных списаний
panel["avg_payment"] = panel["revenue"] / panel["n_payments"]              # средний платёж, ₽
panel["arpu"] = panel["revenue"] / panel["active"]                         # выручка на активную подписку, ₽
panel["revenue_per_day"] = panel["revenue"] / panel["days"]
panel["growth_pct"] = panel["revenue"].pct_change() * 100                  # рост выручки за месяц
panel["growth_per_day_pct"] = panel["revenue_per_day"].pct_change() * 100  # рост выручки в день

COLS = ["active_start", "active", "new_subs", "ended", "churn_pct", "trial_share_pct", "ext_share_pct",
        "n_payments", "failed_pct", "avg_payment", "arpu", "revenue", "revenue_per_day",
        "growth_pct", "growth_per_day_pct"]
panel[COLS].round(1)

In [ ]:
# Разведочная «панель»: четыре маленьких графика по месяцам. Апрель 2026 выделен красным.
# (Разведочный график, в основные графики проекта не входит.)
plot = panel.loc["2025-02":].copy()          # январь 2025 неполный: данные с 6 января
plot["month"] = plot.index
plot["new_per_day"] = plot["new_subs"] / plot["days"]
colors = [ACCENT if month == "2026-04" else GRAY for month in plot.index]

charts = [
    ("revenue_per_day", 1e6, "Выручка в день, млн руб."),
    ("active", 1e3, "Активные подписки в месяце, тыс."),
    ("new_per_day", 1, "Новые подписки в день, шт."),
    ("churn_pct", 1, "Отток за месяц, % от активных на начало"),
]

fig, axes = plt.subplots(2, 2, figsize=(13, 7.5), layout="constrained")
for ax, (col, scale, title) in zip(axes.flat, charts):
    sns.barplot(x=plot["month"], y=plot[col] / scale, hue=plot["month"],
                palette=colors, legend=False, ax=ax)
    ax.set_title(title, loc="left")
    ax.set_xlabel("")
    ax.set_ylabel("")
    ax.set_ylim(bottom=0)                     # ось от нуля
    ax.tick_params(axis="x", rotation=90, labelsize=8)
fig.suptitle("Подписчики в апреле 2026 в порядке: база растёт, отток прежний — проседает только выручка",
             x=0.01, ha="left", fontsize=14)
plt.show()

**Что бросилось в глаза.**

* **Активные подписки** растут каждый месяц на 4–5 тыс.: 175 тыс. в январе 2026, 188 тыс.
  в апреле, 200 тыс. в июле. В апреле рост не остановился.
* **Новые подписки** — ровно около 11 тыс. в месяц (~360 в день), из них ~17 % триалов.
  В феврале меньше только потому, что в нём 28 дней.
* **Отток** — около 4 % активных за месяц всю историю, и **после повышения цен 1 марта
  он не вырос**: март 4,0 %, апрель 3,9 %, май 4,0 %, июнь 3,9 %. Значит, «клиенты ушли
  из-за цены» уже выглядит сомнительно — но это надо проверить по причинам отмены.
* **Доля Расширенного** — 38 % все месяцы; **доля неуспешных списаний** — 3,2–3,4 %,
  в апреле 3,4 %, ничего особенного.
* **Средний платёж** — 275 ₽ до февраля 2026 и 325 ₽ с марта: с 01.03.2026 подняли цены
  (в справочнике `tariffs`: 199 → 249 ₽ и 399 → 449 ₽).
* **Выручка.** Прирост за месяц скачет (февраль 2026 −7,3 %), а в пересчёте на день он
  ровный — короткие месяцы искажают сравнение, сравнивать надо в день (как на С6).
  Март 2026 +21 % в день — это цены. **Апрель 2026 −4,6 % в день** — единственный полный
  месяц, когда выручка в день упала. ARPU в апреле 287 ₽ против 317 ₽ в марте и мае.
  Май +9 % в день — похоже на отскок от низкого апреля.
* Получается странно: **подписчиков в апреле больше, отток прежний, а денег на одну
  подписку пришло меньше.** Это меня и зацепило.
* **Август 2026** — аномалия: отток 18 % вместо 4 % (34,7 тыс. отмен), выручка в день −6,4 %,
  к тому же месяц неполный (история до 30 августа). Ниже посмотрела отмены по дням.
* Январь 2025 неполный с другой стороны (данные с 6 января), поэтому большой рост в феврале
  2025 — артефакт.

In [ ]:
# Август тоже ниже июля. Заодно посмотрела отмены подписок по дням: нет ли выбросов.
ends = q('''
    select s.ended_at::date as day,
           coalesce(c.acquisition_channel, 'без канала') as channel,
           count(*) as ended
    from prime.subscriptions s
    join prime.clients c on c.client_id = s.owner_client_id
    where s.ended_at is not null
    group by 1, 2
''')
ends.sort_values("ended", ascending=False).head(6)
# 22 и 23 августа 2026 за два дня закончилось ~27 тыс. подписок канала partner_telecom
# (и ~600 у клиентов без канала),
# в обычный день по всем каналам — около 250.

### Вопросы, которые у меня возникли

| № | Вопрос | Что видно в данных | Можно ли проверить |
|---|---|---|---|
| 1 | Почему в апреле 2026 упала выручка? | −7,6 % за месяц и −4,6 % в день — единственное падение в день за всю историю, сразу после повышения цен; при этом база растёт и отток не вырос | да: есть полный март и полный апрель, можно разложить по каналам, тарифам и факторам |
| 2 | Сколько принесло повышение цен с 01.03 и не ушли ли клиенты? | средний платёж 274,85 → 324,83 ₽, выручка в день в марте +21 %; отток после 01.03 около 4 %, как и до | да: отмены по причинам до и после 1 марта |
| 3 | Что случилось 22–23.08.2026 с подписками partner_telecom? | за два дня закончилось ~28 тыс. подписок, почти все — канала partner_telecom | частично: история кончается 30.08, влияние на выручку почти не видно |
| 4 | Какой канал привлечения растёт хуже всех? | на семинарах 5–6 у social и performance был самый высокий отток и самый медленный рост базы | да, но это медленный тренд, а не событие |

### Какой вопрос я выбираю и почему

Выбираю **вопрос 1 — почему в апреле 2026 упала выручка**:

* это единственное падение выручки в день за всю историю — то, что руководитель заметит первым;
* оно случилось сразу после повышения цен, и напрашивается решение «откатить цены» — его надо проверить,
  прежде чем принимать; так вопрос 2 входит в мой вопрос как гипотеза;
* есть проверяемые гипотезы и понятное практическое действие по итогам.

Вопрос 3 отложила: август неполный, а платежей после него нет — посчитать влияние на выручку
пока нельзя. Вопрос 4 — про медленный тренд, а не про изменение за период.

До расчётов записала четыре возможные причины апрельского падения — они стали гипотезами ниже:
повышение цен вызвало отток (H1); апрель короче марта (H2); технический сбой с платежами (H3);
падение сидит в одном канале или тарифе (H4).

---

## Бизнес-вопрос

С 1 марта 2026 ПРАЙМ поднял цены: Базовый 199 → 249 ₽, Расширенный 399 → 449 ₽.
В марте выручка от подписок выросла до 58,2 млн ₽, а в апреле упала до 53,8 млн ₽,
то есть на **−4,4 млн ₽ (−7,6 %)**.

Вопрос руководителя: **«Повышение цен начало отпугивать клиентов? Откатывать цены или нет?»**

Мой вопрос: **почему в апреле 2026 упала выручка от подписок — и виновато ли в этом повышение цен?**

## Гипотезы (записала до расчёта)

| № | Гипотеза | Почему возникла | Как проверяю | Раздел |
|---|---|---|---|---|
| H1 | Повышение цен отпугнуло клиентов: вырос отток с причиной «слишком дорого» | падение случилось сразу после повышения цен — первое, что придёт в голову руководителю | отмены `too_expensive` на 1000 активных подписок в день, до и после 1 марта | 4 |
| H2 | Апрель просто короче марта (30 дней против 31) | в разведке увидела, что прирост за месяц скачет вместе с длиной месяца | выручка в пересчёте на день; фактор «дней в месяце» в разложении | 3 и 6 |
| H3 | Технический сбой: часть апрельских платежей не прошла или не попала в базу | в разведке подписчиков в апреле больше, а денег на подписку меньше | платежи по дням; фактор «платежей на подписку в день» в разложении | 6 |
| H4 | Падение сидит в одном сегменте: один канал или бегство на дешёвый тариф | если бы виноват был один канал или тариф, это была бы не «общая» причина | аддитивное разложение по каналам и тарифам, доля Расширенного | 5 |

Порядок проверки: сначала строю витрину и проверяю её (разделы 1–2), смотрю, что именно
изменилось (раздел 3), потом проверяю главную «страшную» гипотезу H1 (раздел 4),
затем ищу, где сидит падение (H4, раздел 5), и за счёт чего оно сложилось (H2 и H3, раздел 6).

## 1. Зерно, период и сегменты

**Зерно: одна строка = месяц × тариф × канал привлечения владельца подписки.**

* **Месяц** — потому что вопрос про месячную выручку (март против апреля), а цена
  и списания устроены помесячно: подписка платит примерно раз в 30 дней.
* **Тариф** — потому что повышение цены было разным: +25 % у Базового и +12,5 %
  у Расширенного. Если клиенты «сбегают» на дешёвый тариф (гипотеза H4), это видно только
  в разрезе тарифа.
* **Канал** — чтобы проверить, не виноват ли один сегмент (H4): если падение сидит в одном
  канале, причина в нём, если везде одинаково — причина общая (цена или техника).
* Мельче (платёж, день) не беру для разложения: на уровне платежа нет «активных подписок»,
  а на уровне дня выручка шумит. Дни понадобятся только для проверки H3.

Канал беру у **владельца** подписки (`subscriptions.owner_client_id`): он и платит.
Клиенты без канала — отдельной строкой «без канала», не выбрасываю: иначе части
не сойдутся с итогом.

**Подписка активна в месяце**, если началась до конца месяца и не закончилась до его начала
(определение с Л5/С6). Выручка — только платежи со статусом `success`.

**Период загрузки** — вся история, январь 2025 — август 2026, чтобы видеть тренд.
Для выводов использую февраль 2025 — июль 2026: январь 2025 неполный (история начинается
6 января), август 2026 тоже неполный (история кончается 30 августа) и в нём есть
отдельная аномалия — о ней в ограничениях.

In [ ]:
# Строю витрину: месяц × тариф × канал -> активные подписки, число успешных платежей, выручка.
# Две части считаю отдельно и соединяю по ключу (месяц, тариф, канал):
#   act — активные подписки (из subscriptions),
#   rev — платежи (из payments).
# subscription_members НЕ присоединяю: платит только владелец, участники
# размножили бы строки платежей (проверка 2.2 ниже).
monthly = q('''
    with act as (
        -- для каждого месяца из generate_series беру подписки, активные в этом месяце
        select to_char(m.month_start, 'YYYY-MM') as month,
               s.tariff_id,
               c.acquisition_channel as channel,
               count(*) as active
        from generate_series(cast(:first as timestamptz), cast(:last as timestamptz),
                             interval '1 month') as m(month_start)
        join prime.subscriptions s
          on s.started_at < m.month_start + interval '1 month'
         and (s.ended_at is null or s.ended_at >= m.month_start)
        join prime.clients c on c.client_id = s.owner_client_id
        group by 1, 2, 3
    ),
    rev as (
        -- успешные платежи: подписка -> владелец -> его канал
        select to_char(p.paid_at, 'YYYY-MM') as month,
               s.tariff_id,
               c.acquisition_channel as channel,
               count(*) as n_payments,
               sum(p.amount) as revenue
        from prime.payments p
        join prime.subscriptions s on s.subscription_id = p.subscription_id
        join prime.clients c on c.client_id = s.owner_client_id
        where p.status = 'success'
          and p.paid_at >= cast(:first as timestamptz)
          and p.paid_at <  cast(:last as timestamptz) + interval '1 month'
        group by 1, 2, 3
    )
    select a.month, a.tariff_id, a.channel, a.active,
           coalesce(r.n_payments, 0) as n_payments,
           coalesce(r.revenue, 0)    as revenue
    from act a
    left join rev r
      on r.month = a.month
     and r.tariff_id = a.tariff_id
     and r.channel is not distinct from a.channel   -- NULL = NULL для «без канала»
    order by 1, 2, 3
''', first="2025-01-01", last="2026-08-01")

# Пропуск канала -> отдельная строка «без канала» (договорённость курса).
monthly["channel"] = monthly["channel"].fillna("без канала")
monthly["tariff"] = monthly["tariff_id"].map({1: "Базовый", 2: "Расширенный"})

# Зерно должно быть уникальным: одна строка на месяц × тариф × канал.
assert not monthly.duplicated(["month", "tariff_id", "channel"]).any(), "зерно не уникально"
assert monthly["channel"].isna().sum() == 0

print(len(monthly), "строк =", monthly["month"].nunique(), "месяцев ×",
      monthly["tariff_id"].nunique(), "тарифа ×", monthly["channel"].nunique(), "каналов")
monthly.head(7)

## 2. Проверки перед расчётом

Прежде чем проверять гипотезы, убеждаюсь, что витрина считает правильно. В описании базы
три ловушки: двойной счёт через `subscription_members`, справочник тарифов с историей
и потерянные строки. Если витрина врёт, все выводы ниже — тоже.

In [ ]:
# 2.1. Сверка: выручка в витрине = выручка прямо из payments (без соединений).
# Если соединения что-то потеряли или задвоили, суммы разойдутся.
direct = q('''
    select sum(amount) as revenue, count(*) as n_payments
    from prime.payments
    where status = 'success'
      and paid_at >= '2025-01-01' and paid_at < '2026-09-01'
''').iloc[0]

print("витрина:      ", rub(monthly["revenue"].sum()), "₽,", rub(monthly["n_payments"].sum()), "платежей")
print("прямой запрос:", rub(direct["revenue"]), "₽,", rub(direct["n_payments"]), "платежей")
assert abs(monthly["revenue"].sum() - float(direct["revenue"])) < 1, "витрина потеряла или задвоила выручку"
assert monthly["n_payments"].sum() == direct["n_payments"]
# Сошлось до рубля — витрине можно верить.

In [ ]:
# 2.2. Двойной счёт: что было бы, если присоединить subscription_members к платежам.
# Март–апрель 2026, только успешные платежи.
dbl = q('''
    select
      (select count(*) from prime.payments p
        where p.status = 'success' and p.paid_at >= '2026-03-01' and p.paid_at < '2026-05-01') as payments,
      (select count(*) from prime.payments p
        join prime.subscription_members sm on sm.subscription_id = p.subscription_id
        where p.status = 'success' and p.paid_at >= '2026-03-01' and p.paid_at < '2026-05-01') as with_members
''').iloc[0]

print("платежей:                    ", rub(dbl["payments"]))
print("строк после join с участниками:", rub(dbl["with_members"]),
      f"(в {dbl['with_members'] / dbl['payments']:.2f} раза больше)")
# Строк стало больше: каждый платёж повторился столько раз, сколько у подписки участников,
# а платежи подписок без участников вообще пропали бы. Поэтому в витрине этого join нет.

In [ ]:
# 2.3. Справочник тарифов: у одного tariff_id несколько строк с интервалами цен.
tariffs = q("select * from prime.tariffs order by tariff_id, valid_from")
print(tariffs[["tariff_id", "tariff_name", "monthly_fee", "valid_from", "valid_to"]].to_string(index=False))

# У Расширенного интервалы пересекаются: строка 429 ₽ (02.2026–06.2026) накрывает
# и 399 ₽, и 449 ₽. Какая цена реально списывалась? Смотрю на сами платежи.
amounts = q('''
    select to_char(p.paid_at, 'YYYY-MM') as month, s.tariff_id, p.amount, count(*) as n
    from prime.payments p
    join prime.subscriptions s on s.subscription_id = p.subscription_id
    where p.status = 'success' and p.paid_at >= '2026-01-01' and p.paid_at < '2026-08-01'
    group by 1, 2, 3 order by 1, 2, 3
''')
print()
# pivot упадёт, если в месяце у тарифа больше одной суммы — это тоже проверка.
print(amounts.pivot(index="month", columns="tariff_id", values="amount"))
# 429 ₽ не списывалось ни разу: строка в справочнике ошибочная. Если бы я брала цену
# из справочника по дате, платежи Расширенного за март–июнь задвоились бы.
# Поэтому цену беру из самих платежей: price = revenue / n_payments.

In [ ]:
# 2.4. Нет ли дней совсем без платежей? (сутки по UTC)
# Беру все дни истории и присоединяю к ним число платежей; пустой день даст 0.
gaps = q('''
    with d as (select generate_series(date '2025-01-06', date '2026-08-30', interval '1 day')::date as day),
         p as (select paid_at::date as day, count(*) as n from prime.payments group by 1)
    select d.day, coalesce(p.n, 0) as n_payments
    from d left join p on p.day = d.day
    where coalesce(p.n, 0) < 3000          -- в обычный день 5–6 тысяч платежей
    order by 1
''')
gaps
# 06.01.2025 и 30.08.2026 — края истории, неполные сутки.
# 14 и 15 апреля 2026 — НОЛЬ платежей, даже неуспешных. Запомнила: это к гипотезе H3.

## 3. Что случилось: выручка по месяцам и в пересчёте на день

Сначала точно фиксирую, что именно изменилось между мартом и апрелем, — и сразу первый
взгляд на **H2** (короткий месяц): если всё дело в длине месяца, то в пересчёте на день
апрель не должен быть ниже марта.

In [ ]:
# Свожу витрину до одной строки на месяц.
m = monthly.groupby("month")[["active", "n_payments", "revenue"]].sum()
m["days"] = pd.PeriodIndex(m.index, freq="M").days_in_month     # 28–31 день
m["revenue_per_day"] = m["revenue"] / m["days"]                  # выручка в день
m["growth_pct"] = m["revenue"].pct_change() * 100                # прирост за месяц, %
m["growth_per_day_pct"] = m["revenue_per_day"].pct_change() * 100  # прирост в день, %

m.loc["2025-12":"2026-08", ["active", "n_payments", "revenue", "days",
                            "revenue_per_day", "growth_pct", "growth_per_day_pct"]].round(1)

In [ ]:
A, B = "2026-03", "2026-04"   # сравниваю март и апрель 2026
total_delta = m.loc[B, "revenue"] - m.loc[A, "revenue"]
print(f"выручка: {rub(m.loc[A, 'revenue'])} -> {rub(m.loc[B, 'revenue'])} ₽, "
      f"изменение {rub(total_delta)} ₽ ({m.loc[B, 'growth_pct']:+.1f} %)")
print(f"в день:  {rub(m.loc[A, 'revenue_per_day'])} -> {rub(m.loc[B, 'revenue_per_day'])} ₽, "
      f"{m.loc[B, 'growth_per_day_pct']:+.1f} %")
# Даже в пересчёте на день апрель ниже марта. Значит, длиной месяца (H2) всё не объяснить.

In [ ]:
# График 1. Выручка в день по месяцам. Период выводов: 02.2025–07.2026.
plot = m.loc["2025-02":"2026-07"].reset_index()
plot["revenue_mln"] = plot["revenue_per_day"] / 1e6
colors = [ACCENT if month == B else GRAY for month in plot["month"]]   # апрель — красным

fig, ax = plt.subplots(figsize=(11, 4.6), layout="constrained")
sns.barplot(data=plot, x="month", y="revenue_mln", hue="month", palette=colors, legend=False, ax=ax)
x_march = list(plot["month"]).index("2026-03") - 0.5
ax.axvline(x_march, color="#111827", linestyle="--", linewidth=0.8)
ax.text(x_march + 0.1, 2.15, "повышение цен 01.03", fontsize=10)
ax.set_title("Апрель 2026 — единственный месяц, когда выручка в день упала", loc="left")
ax.set_ylabel("выручка в день, млн руб.")
ax.set_xlabel("месяц (UTC); выручка месяца / число дней в месяце")
ax.set_ylim(0, 2.35)                       # ось от нуля
ax.tick_params(axis="x", rotation=60)
plt.show()

**Результат.** Выручка: 58,2 → 53,8 млн ₽, **−4,4 млн ₽ (−7,6 %)**. В пересчёте на день:
1,88 → 1,79 млн ₽, **−4,6 %**.

**Промежуточный вывод по H2.** Короткий месяц объясняет не всё: даже в пересчёте на день
апрель ниже марта. Сколько именно рублей дала длина месяца, посчитаю в разложении (раздел 6).

## 4. Гипотеза H1: повышение цен отпугнуло клиентов

**Гипотеза.** После повышения цен 1 марта клиенты стали чаще отменять подписку из-за цены,
база начала таять — и выручка в апреле упала.

**Почему возникла.** Падение случилось ровно через месяц после повышения цен. Это первое,
что подумает руководитель, и от этой гипотезы зависит решение «откатывать цены или нет».

**Как проверяю.** В `subscriptions` у каждой отмены есть причина `cancel_reason`. Считаю,
сколько отмен с причиной `too_expensive` приходится **на 1000 активных подписок в день**,
по месяцам с сентября 2025 по июль 2026 (полгода до и почти полгода после повышения).
Нормирую на активные подписки, потому что база растёт, и на дни — потому что месяцы разной длины;
без этого рост числа отмен ничего не значит. Август исключаю — там отдельная аномалия.

**Что ожидаю, если H1 верна.** С марта 2026 показатель заметно выше, чем до марта.

In [ ]:
# Шаг 1. Число отмен по месяцам: всего и с причиной «слишком дорого».
churn = q('''
    select to_char(ended_at, 'YYYY-MM') as month,
           count(*) filter (where cancel_reason = 'too_expensive') as too_expensive,
           count(*) as ended
    from prime.subscriptions
    where ended_at >= '2025-09-01' and ended_at < '2026-08-01'
    group by 1 order by 1
''').set_index("month")

# Шаг 2. Нормирую: отмен на 1000 активных подписок в день.
churn = churn.join(m[["active", "days"]])
churn["expensive_per_1000_day"] = churn["too_expensive"] / churn["active"] / churn["days"] * 1000
churn["all_per_1000_day"] = churn["ended"] / churn["active"] / churn["days"] * 1000

# Шаг 3. Помечаю месяцы после повышения цен и сравниваю средние «до» и «после».
churn["after"] = churn.index >= "2026-03"

print(churn.round(3))
print()
print(churn.groupby("after")[["expensive_per_1000_day", "all_per_1000_day"]].mean()
      .rename(index={False: "до 03.2026", True: "с 03.2026"}).round(3))

In [ ]:
# Ещё одна проверка H1: если клиенты бегут от цены, они могли перейти на дешёвый тариф.
# Тогда доля Расширенного в платежах после марта упала бы.
pay_t = monthly.pivot_table(index="month", columns="tariff", values="n_payments", aggfunc="sum")
share_ext = (pay_t["Расширенный"] / pay_t.sum(axis=1) * 100).loc["2025-12":"2026-07"].round(1)
print("доля Расширенного в платежах, %:")
print(share_ext.to_string())

In [ ]:
# График 2. Отток «дорого» до и после повышения цен.
plot = churn.reset_index()
plot["month_dt"] = pd.to_datetime(plot["month"])   # даты, чтобы обе линии легли на одну ось

fig, ax = plt.subplots(figsize=(11, 4.4), layout="constrained")
sns.lineplot(data=plot, x="month_dt", y="expensive_per_1000_day", color=GRAY, marker="o", ax=ax)
sns.lineplot(data=plot[plot["after"]], x="month_dt", y="expensive_per_1000_day", color=ACCENT,
             marker="o", linewidth=2.5, label="после повышения цен", ax=ax)
ax.axvline(pd.Timestamp("2026-02-15"), color="#111827", linestyle="--", linewidth=0.8)
ax.set_title("После повышения цен отмен «слишком дорого» не стало больше — H1 не подтвердилась", loc="left")
ax.set_ylabel("отмен too_expensive\nна 1000 активных подписок в день")
ax.set_xlabel("месяц отмены (UTC)")
ax.set_ylim(0, 0.4)                        # ось от нуля
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))
ax.legend(frameon=False, loc="lower right")
plt.show()

**Результат.**

* Отмен «слишком дорого»: **0,296** на 1000 активных подписок в день до марта и **0,293** после.
  В апреле — 0,297, как в любой другой месяц.
* Все отмены вместе: 1,225 до и 1,213 после — тоже без роста.
* Доля Расширенного в платежах — 37,9–38,0 % и до, и после: на дешёвый тариф никто не ушёл.
* А база активных подписок в апреле **выросла**: 183,8 → 187,6 тыс.

**Вывод: H1 не подтвердилась.** Если бы цена отпугивала клиентов, отмен «дорого» стало бы больше
и база перестала бы расти. Ни того, ни другого нет.

**Что дальше.** Раз клиенты не ушли, падение надо искать в другом месте. Сначала проверю, не сидит
ли оно в каком-то одном канале или тарифе (H4).

## 5. Гипотеза H4: падение сидит в одном сегменте

**Гипотеза.** Выручка упала не везде, а в каком-то одном канале привлечения или тарифе
(например, отвалился партнёрский канал).

**Почему возникла.** Если причина в одном сегменте, это совсем другая история и другие
действия, чем если упало всё сразу.

**Как проверяю.** Аддитивное разложение: выручка = сумма выручек каналов (и тарифов),
значит изменение выручки = сумма изменений по каналам. Считаю изменение каждого
канала в рублях и в процентах **в пересчёте на день** (чтобы убрать разницу 31 и 30 дней).
Проверка: сумма изменений частей должна равняться изменению итога — `assert`.

**Что ожидаю, если H4 верна.** Один-два канала (или один тариф) падают сильно, остальные — нет.

In [ ]:
# Шаг 1. Каналы — в строки, месяцы — в колонки, в клетке сумма выручки по двум тарифам.
by_channel = monthly.pivot_table(index="channel", columns="month", values="revenue", aggfunc="sum")
by_channel = by_channel[[A, B]].copy()
by_channel["delta"] = by_channel[B] - by_channel[A]                 # изменение в рублях
# Шаг 2. Изменение выручки в день: апрель делю на 30 дней, март — на 31.
by_channel["per_day_pct"] = ((by_channel[B] / 30) / (by_channel[A] / 31) - 1) * 100
by_channel = by_channel.sort_values("delta")

# Шаг 3. Сумма частей = итог (до копейки). Если не сходится — потеряла строки.
assert abs(by_channel["delta"].sum() - total_delta) < 0.01, "каналы не сходятся с итогом"

summary = by_channel.copy()
summary.loc["итого"] = [summary[A].sum(), summary[B].sum(), summary["delta"].sum(),
                        ((summary[B].sum() / 30) / (summary[A].sum() / 31) - 1) * 100]
summary.round({A: 0, B: 0, "delta": 0, "per_day_pct": 1})

In [ ]:
# То же разложение по тарифам.
by_tariff = monthly.pivot_table(index="tariff", columns="month", values="revenue", aggfunc="sum")[[A, B]]
by_tariff["delta"] = by_tariff[B] - by_tariff[A]
by_tariff["per_day_pct"] = ((by_tariff[B] / 30) / (by_tariff[A] / 31) - 1) * 100
assert abs(by_tariff["delta"].sum() - total_delta) < 0.01, "тарифы не сходятся с итогом"

by_tariff.round({A: 0, B: 0, "delta": 0, "per_day_pct": 1})

In [ ]:
# График 3. Изменение выручки в день по каналам: одинаковое везде.
plot = by_channel.sort_values("per_day_pct").reset_index()
total_pct = summary.loc["итого", "per_day_pct"]

fig, ax = plt.subplots(figsize=(10, 4.4), layout="constrained")
sns.barplot(data=plot, y="channel", x="per_day_pct", color=GRAY, ax=ax)
ax.axvline(total_pct, color=ACCENT, linewidth=2.5,
           label=f"все каналы: {total_pct:.1f} %".replace(".", ",").replace("-", "−"))
ax.axvline(0, color="#111827", linewidth=0.8)
ax.set_title("Выручка в день упала во всех каналах почти одинаково — дело не в сегменте", loc="left")
ax.set_xlabel("изменение выручки в день, апрель 2026 к марту 2026, %")
ax.set_ylabel("канал привлечения владельца")
ax.legend(frameon=False, loc="lower left")
plt.show()

**Результат.**

* Падение есть во **всех семи** строках каналов, включая «без канала»: от −2,9 до −5,7 % в день
  при общем −4,6 %. Больше всех в рублях потерял organic (−1,17 млн), но это просто самый
  большой канал — в процентах он падает как все (−4,0 %).
* Оба тарифа падают одинаково: Базовый −4,7 % в день, Расширенный −4,4 %.
* Сумма изменений по каналам и по тарифам равна изменению итога (−4 442 505 ₽) — `assert` прошёл.

**Вывод: H4 не подтвердилась.** Падение не сидит в одном сегменте — задело всех сразу и одинаково.
Такое бывает, когда причина общая: календарь (H2) или техника (H3).

**Что дальше.** Раскладываю изменение выручки на множители, чтобы разделить H2 и H3 в рублях.

## 6. Гипотезы H2 и H3: короткий месяц и пропавшие платежи

**Гипотезы.**
* **H2** — апрель короче марта на один день, поэтому платежей меньше.
* **H3** — технический сбой: часть апрельских платежей не прошла или не попала в базу.

**Почему возникли.** H2 — из разведки: прирост за месяц скачет вместе с длиной месяца.
H3 — из разведки и проверки 2.4: подписчиков в апреле больше, отток прежний, а денег на подписку
меньше; и 14–15 апреля в базе нет ни одного платежа.

**Как проверяю.** Мультипликативная модель — выручка раскладывается на множители:

> **выручка = активные × дней в месяце × платежей на подписку в день × цена платежа**

Произведение четырёх множителей равно выручке ровно (дни сокращаются). Каждый множитель
отвечает за свою гипотезу:

* «активные подписки» — база (это ещё раз про H1: если клиенты ушли, тут будет минус);
* «дней в месяце» — это **H2**;
* «платежей на подписку в день» — как часто подписка платит; списание идёт раз в 30 дней,
  сама по себе частота меняться не должна — если она упала, это **H3**;
* «цена платежа» — цены в марте и апреле одинаковые, тут жду ноль.

Влияние каждого множителя считаю цепными подстановками (семинар 6): меняю факторы по одному
в порядке «сначала количественные, потом качественные»: активные → дни → платежей в день → цена.

**Что ожидаю.** Если верна H2 — минус около 1/31 выручки от фактора «дни» (≈ −1,9 млн).
Если верна H3 — заметный минус от фактора «платежей на подписку в день», и его размер
должен совпасть с тем, сколько платежей «должно было быть» 14–15 апреля.

In [ ]:
def chain(n0, p0, n1, p1):
    # Цепные подстановки для R = N × P: сначала количество (при старой цене), потом цена.
    by_count = (n1 - n0) * p0
    by_price = n1 * (p1 - p0)
    # Проверка: сумма влияний = изменение показателя.
    assert abs(by_count + by_price - (n1 * p1 - n0 * p0)) < 0.01, "сумма влияний не равна изменению"
    return round(by_count), round(by_price)


def contrib(x0: dict, x1: dict, order: list[str]) -> dict:
    # Влияние каждого фактора: меняю факторы по одному в порядке order,
    # остальные держу на текущем уровне.
    cur, out = dict(x0), {}
    for f in order:
        before = math.prod(cur.values())
        cur[f] = x1[f]                                   # фактор переходит на новое значение
        out[f] = math.prod(cur.values()) - before        # насколько от этого изменился итог
    total = math.prod(x1.values()) - math.prod(x0.values())
    assert abs(sum(out.values()) - total) < 0.01, "сумма влияний не равна изменению"
    return out


# Проверяю функции на примерах с лекции, где ответ известен заранее.
# Пекарня: 100 булок по 50 ₽ -> 90 булок по 60 ₽. Ждём (-500, 900).
print(chain(100, 50, 90, 60))
# Комиссия: ждём 3000, -3300, 1980.
toy = contrib({"ops": 1000, "chk": 2000, "rate": 0.015},
              {"ops": 1100, "chk": 1800, "rate": 0.016}, ["ops", "chk", "rate"])
print({f: round(v) for f, v in toy.items()})

In [ ]:
# Шаг 1. Считаю четыре множителя для каждого месяца.
FACTORS = ["active", "days", "per_sub_day", "price"]
LABELS = {"active": "активные подписки", "days": "дней в месяце",
          "per_sub_day": "платежей на подписку в день", "price": "цена платежа"}

m["per_sub_day"] = m["n_payments"] / m["active"] / m["days"]   # как часто платит одна подписка
m["price"] = m["revenue"] / m["n_payments"]                     # цена одного платежа

# Произведение факторов должно давать выручку каждого месяца.
check = m[FACTORS].prod(axis=1)
assert (check - m["revenue"]).abs().max() < 0.01, "факторы не перемножаются в выручку"

m.loc["2025-12":"2026-07", FACTORS].round(5)
# «платежей на подписку в день» держится около 0,0314 (раз в ~30 дней)
# и только в апреле 2026 проседает до 0,0294.

In [ ]:
# Шаг 2. Разложение март -> апрель цепными подстановками.
x0 = m.loc[A, FACTORS].to_dict()   # март
x1 = m.loc[B, FACTORS].to_dict()   # апрель

effects = pd.Series(contrib(x0, x1, FACTORS))
assert abs(effects.sum() - total_delta) < 0.01, "сумма влияний не равна изменению выручки"

print(effects.rename(LABELS).map(rub).to_string())
print(f"\nсумма влияний {rub(effects.sum())} ₽ = изменение выручки {rub(total_delta)} ₽")

# Для сравнения — двухфакторная модель с С6: база × выручка на подписку.
by_active, by_rev_per_sub = chain(m.loc[A, "active"], m.loc[A, "revenue"] / m.loc[A, "active"],
                                  m.loc[B, "active"], m.loc[B, "revenue"] / m.loc[B, "active"])
print(f"\nдва фактора: база {rub(by_active)} ₽, выручка на подписку {rub(by_rev_per_sub)} ₽")

**Результат разложения.**

| Фактор | Влияние | Какая гипотеза |
|---|---:|---|
| активные подписки | **+1,20 млн ₽** | H1: база выросла — клиенты не ушли |
| дней в месяце | **−1,92 млн ₽** | H2 |
| платежей на подписку в день | **−3,75 млн ₽** | H3 |
| цена платежа | +0,02 млн ₽ | цены не менялись |
| **итого** | **−4,44 млн ₽** | |

**Вывод по H2: подтвердилась частично.** Короткий месяц действительно отнял −1,9 млн ₽ —
но это меньше половины падения.

Главный минус — **частота платежей**. Подписка платит раз в 30 дней, сама по себе частота
не меняется. Проверяю H3 напрямую: сколько платежей пропало 14–15 апреля и сходится ли это
с −3,75 млн ₽.

In [ ]:
# Шаг 1. Платежи по дням за март–май 2026.
daily = q('''
    select paid_at::date as day, count(*) as n_payments, sum(amount) as revenue
    from prime.payments
    where status = 'success' and paid_at >= '2026-03-01' and paid_at < '2026-06-01'
    group by 1 order by 1
''')
daily["day"] = pd.to_datetime(daily["day"])
daily = daily.set_index("day").asfreq("D", fill_value=0)   # пустые дни -> явные нули

# Шаг 2. Сколько платежей было бы 14–15 апреля, если бы это были обычные дни:
# беру среднее по остальным дням апреля и умножаю на 2.
april = daily.loc["2026-04"]
normal_days = april[april["n_payments"] > 0]
expected_pay = normal_days["n_payments"].mean() * 2
expected_rev = expected_pay * m.loc[B, "price"]          # в рублях по апрельской цене

print(f"обычный апрельский день: {rub(normal_days['n_payments'].mean())} платежей")
print(f"ожидалось 14–15 апреля:  ~{rub(expected_pay)} платежей ≈ {rub(expected_rev)} ₽")
print(f"влияние фактора «платежей в день» в разложении: {rub(effects['per_sub_day'])} ₽")

# Шаг 3. Контрольный расчёт: если вернуть эти два дня, какой была бы выручка в день в апреле?
fixed = (m.loc[B, "revenue"] + expected_rev) / 30
print(f"\nапрель с двумя днями: {rub(fixed)} ₽ в день против {rub(m.loc[A, 'revenue_per_day'])} в марте "
      f"({(fixed / m.loc[A, 'revenue_per_day'] - 1) * 100:+.1f} %)")

In [ ]:
# График 4. Платежи по дням, март–май 2026.
plot = daily.reset_index()
plot["n_thousand"] = plot["n_payments"] / 1000
zero = plot[plot["n_payments"] == 0]          # пустые дни — отмечу красным

fig, ax = plt.subplots(figsize=(11, 4.4), layout="constrained")
sns.lineplot(data=plot, x="day", y="n_thousand", color=GRAY, linewidth=1.8, ax=ax)
sns.scatterplot(data=zero, x="day", y="n_thousand", color=ACCENT, s=90, zorder=3, ax=ax)
ax.annotate("14–15 апреля: 0 платежей", xy=(pd.Timestamp("2026-04-15"), 0.2),
            xytext=(pd.Timestamp("2026-04-22"), 2.0), color=ACCENT, fontsize=11,
            arrowprops={"arrowstyle": "->", "color": ACCENT})
ax.set_title("14 и 15 апреля 2026 не прошло ни одного платежа — обычно их ~5,9 тыс. в день", loc="left")
ax.set_ylabel("успешных платежей, тыс. в день")
ax.set_xlabel("день (UTC), март–май 2026")
ax.set_ylim(0, 7)                          # ось от нуля
plt.show()

In [ ]:
def waterfall(deltas: pd.Series, title: str, accent: str) -> plt.Axes:
    # Каскад изменений: столбики факторов «плывут» (каждый начинается там,
    # где закончился предыдущий), последний — «итого» от нуля.
    before = deltas.cumsum() - deltas
    colors = [ACCENT if name == accent else GRAY for name in deltas.index]

    fig, ax = plt.subplots(figsize=(10, 4.6), layout="constrained")
    bars = ax.bar(deltas.index, deltas, bottom=before, color=colors, zorder=2)
    total = ax.bar("итого", deltas.sum(), color="#4B5563", zorder=2)
    for group in (bars, total):
        ax.bar_label(group, fmt=lambda v: f"{v:+,.0f}".replace(",", " ").replace("-", "−"), padding=3)
    ax.axhline(0, color="#111827", linewidth=0.8)
    ax.set_title(title, loc="left")
    ax.set_ylabel("тыс. руб.")
    ax.use_sticky_edges = False
    ax.margins(y=0.15)
    ax.spines[["top", "right"]].set_visible(False)
    ax.yaxis.set_major_formatter(THOUSANDS)   # 1 000 вместо 1000
    ax.grid(axis="x", visible=False)          # вертикальные линии сетки мешают столбикам
    return ax


# График 5. Каскад разложения март -> апрель.
# Подписи переношу на две строки, чтобы не налезали друг на друга.
steps = effects.rename({f: LABELS[f].replace(" ", "\n", 1) for f in FACTORS}) / 1000
ax = waterfall(steps, "Выручку уронили пропавшие платежи, а не уход клиентов: база добавила +1,2 млн",
               accent=LABELS["per_sub_day"].replace(" ", "\n", 1))
ax.set_xlabel("Март → апрель 2026, тыс. руб. Порядок подстановок — слева направо.")
plt.show()

**Результат.**

* 14 и 15 апреля 2026 в `payments` **ноль записей** — ни успешных, ни неуспешных.
  В обычный апрельский день — около 5,9 тыс. платежей.
* Если бы это были обычные дни, было бы ~11,8 тыс. платежей ≈ **3,84 млн ₽**.
* Фактор «платежей на подписку в день» в разложении — **−3,75 млн ₽**. Числа почти совпадают.
* Если вернуть два дня, выручка в день в апреле была бы **на 2,3 % выше мартовской** —
  обычный темп роста, как в другие месяцы.

**Вывод: H3 подтвердилась.** Основная часть падения (−3,75 из −4,44 млн ₽) — это два дня,
за которые в базе нет ни одного платежа. Чего данные сказать не могут: списаний правда не было
(сбой биллинга) или они были, но не загрузились в базу. Это надо сверять с платёжной командой.

## 7. Итог по гипотезам

| № | Гипотеза | Результат | Вердикт |
|---|---|---|---|
| H1 | Повышение цен отпугнуло клиентов | отмены «дорого» 0,296 → 0,293 на 1000 активных в день; база выросла на 3,8 тыс.; доля Расширенного 38 % без изменений | ❌ не подтвердилась |
| H4 | Падение в одном сегменте | все 7 каналов −2,9…−5,7 % в день, оба тарифа −4,4…−4,7 % | ❌ не подтвердилась |
| H2 | Апрель короче марта | фактор «дней в месяце» −1,92 млн ₽, но и в день апрель ниже марта на −4,6 % | ◐ подтвердилась частично |
| H3 | Сбой: пропали платежи | 14–15 апреля 0 платежей; ожидалось ~3,84 млн ₽; фактор частоты −3,75 млн ₽ | ✅ подтвердилась |

## 8. Вывод и рекомендация

**Что случилось.** Выручка от подписок в апреле 2026 — 53,8 млн ₽ против 58,2 млн ₽ в марте:
−4,4 млн ₽ (−7,6 %). В пересчёте на день — −4,6 %. Это единственный месяц за полтора года,
когда выручка в день упала.

**За счёт чего** (цепные подстановки, март → апрель):

* активные подписки: **+1,2 млн ₽** — база продолжила расти;
* дней в месяце (30 против 31): **−1,9 млн ₽** — эффект календаря (H2);
* платежей на подписку в день: **−3,8 млн ₽** — 14 и 15 апреля в базе нет ни одного
  платежа, ни успешного, ни неуспешного (H3). Ожидалось ~11,8 тыс. платежей ≈ 3,8 млн ₽;
* цена платежа: +0,02 млн ₽, то есть ≈ 0 — цены в марте и апреле одинаковые.

Падение одинаковое во всех каналах и обоих тарифах (H4 не подтвердилась).

**Повышение цен ни при чём (H1 не подтвердилась).** Отток с причиной «дорого» после повышения цен
остался на уровне ~0,3 отмены на 1000 активных подписок в день, общий отток тоже не вырос.
Без двух пустых дней апрельская выручка в день была бы на ~2,3 % выше мартовской — обычный темп роста.

**Рекомендация.**

1. **Цены не откатывать.** Повышение дало +18 % к цене платежа и не увеличило отток.
2. **Платёжной команде и команде данных — до 06.11.2026** сверить 14–15 апреля 2026
   с выпиской эквайринга: если списаний не было — это ~11,8 тыс. подписок и ~3,8 млн ₽
   недополученной выручки (повторно списать или проверить, что они заплатили позже);
   если списания были, но не попали в базу — дозагрузить, иначе отчётность занижена.
3. **Аналитике** — добавить в ежедневный мониторинг проверку «день без платежей»
   (как проверка 2.4 выше), чтобы такой сбой ловился в тот же день, а не через месяц.

**Ограничения.**

* По данным нельзя отличить «платежи не прошли» от «платежи не загрузились»: нужна сверка
  с биллингом.
* Оценка 3,8 млн ₽ — по среднему апрельскому дню; реальный график списаний мог отличаться.
* Причина отмены `cancel_reason` — то, что выбрал клиент; часть «дорогих» могла выбрать «другое».
* Сутки и месяцы по UTC, не по местному времени клиента.
* Август 2026 не анализировался: история кончается 30 августа, а 22–23 августа одномоментно
  закончилось ~28 тыс. подписок, почти все — канала `partner_telecom`. Это отдельный вопрос.

---

🥚 *Если вы или ваш Клодик дочитали до сюда — вот анекдот.*

Живут муж с женой в деревне. Муж пьёт, прям не просыхает. Жена идёт к соседке за советом:
«Муж пьёт, что только не пробовала — никак не могу отучить. А твой, я слышала, бросил.
Как ты его отучила?» — «Да очень просто. Опустила дохлую кошку в бочку с брагой. Муж допил,
вынул кошку, решил, что у него белочка, — и перестал пить».

Жена так и сделала: нашла дохлую кошку, бросила в бочку с брагой. Муж пьёт день, второй —
брага не кончается. На третий день жена приходит домой и видит картину: муж сидит за столом,
выжимает кошку и приговаривает: «Мусечка, ну ещё капельку…»